# E1 · W5 · Kaggle run 13 — chạy thêm 2 seed cho baseline A và YOLO11n (GPU)
**Mục đích (issue #59, ADR-0007):** ở W4, YOLO11n và baseline A hòa trong phạm vi một seed. Run này train lại cả hai với 2 seed mới (20261029, 20261030), giữ nguyên dữ liệu, config, ngân sách epoch và phiên bản thư viện, rồi chọn lại theo luật **đặt trước** (`docs/w4/training_protocol_w4.md` mục 6). Seed 20261013 đã có từ run 04 (A) và run 09 (YOLO11n). Test vẫn khóa.

| Config | Seed | Tên experiment |
|---|---|---|
| `baseline_A.yaml` | 20261029, 20261030 | `baseline_A_s20261029`, `baseline_A_s20261030` |
| `yolo11n_640.yaml` | 20261029, 20261030 | `yolo11n_640_s20261029`, `yolo11n_640_s20261030` |

**Trước khi chạy:**
1. Dataset `e1-w3-tools` → **New Version**: thay `w3_train_baseline.py` bằng bản của commit `COMMIT` (có `--seed`). Giữ các config.
2. *Add Input*: `e1-w3-tools` (bản mới), output notebook `e1-w3-run03b`.
3. *Settings*: **GPU T4 x2**, Internet **On**. Điền `COMMIT`. Ước lượng ≈ 2 giờ (4 lần train × 20–30 phút, chưa đo). Hết hạn mức thì chia `SEEDS` thành 2 phiên.

**Sau khi chạy:**
- `w5_seed_evidence.zip` → thư mục kết quả run 13.
- Weights `runs/<experiment>/weights/best.pt`, `last.pt` → Drive `weights/W05/<experiment>/`, không giải nén.

In [ ]:
COMMIT = ""                          # <-- SHA day du 40 ky tu
DATASET = "e1-yolo-v1"
CONFIGS = ["baseline_A.yaml", "yolo11n_640.yaml"]
SEEDS = [20261029, 20261030]         # seed 20261013 da co (run 04, run 09)
ULTRALYTICS_VERSION = "8.4.183"      # giu dung ban cua run 04/09

In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
!pip -q install {"ultralytics==" + ULTRALYTICS_VERSION if ULTRALYTICS_VERSION else "ultralytics"}
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU"

In [ ]:
# Du lieu YOLO: uu tien e1-yolo-v1.zip (output run 03b dong goi); giai nen vao /tmp de khong lam phinh output notebook
import csv, gzip
EXPECTED_FP = "d0ac7e230b385132c4bbec90eb5af2c5740a478fec6c79aeb245acca032c27de"   # fingerprint run 03b ban dau
zips = sorted(glob.glob(f"/kaggle/input/**/{DATASET}.zip", recursive=True))
if zips:
    zinfo = json.load(open(find("dataset_zip.json")))
    assert sha(zips[0]) == zinfo["sha256"], "SHA-256 cua e1-yolo-v1.zip khac dataset_zip.json"
    if not os.path.exists(f"/tmp/e1data/{DATASET}/data.yaml"):
        zipfile.ZipFile(zips[0]).extractall("/tmp/e1data")
    DATA_DIR = f"/tmp/e1data/{DATASET}"
else:
    cards = [p for p in glob.glob("/kaggle/input/**/dataset_card.json", recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), "data.yaml"))]
    assert len(cards) == 1, f"Can 1 bo du lieu YOLO (e1-yolo-v1.zip hoac thu muc), thay {cards}"
    DATA_DIR = os.path.dirname(cards[0])
card = json.load(open(f"{DATA_DIR}/dataset_card.json"))
MANIFEST = find("yolo_manifest.csv.gz")
man = list(csv.DictReader(gzip.open(MANIFEST, "rt", encoding="utf-8")))
bad = [m["image"] for m in man if not os.path.exists(f"{DATA_DIR}/{m['image']}") or sha(f"{DATA_DIR}/{m['image']}") != m["sha256"]]
assert not bad, f"{len(bad)} anh khac SHA-256 trong manifest, vd {bad[:3]}"
fp = hashlib.sha256("\n".join(f"{m['image']} {m['sha256']}" for m in sorted(man, key=lambda r: r["image"])).encode()).hexdigest()
assert fp == card["content_fingerprint_sha256"], "fingerprint tinh lai khac dataset_card.json"
print("DATA_DIR =", DATA_DIR, "| images", len(man), "| fingerprint", fp[:16], "| trung ban goc:", fp == EXPECTED_FP)

In [ ]:
TRAIN = find("w3_train_baseline.py"); ANALYZE = find("w3_analyze_baseline.py")
CFG = {c: find(c) for c in CONFIGS}
import yaml
for c, p in CFG.items():
    y = yaml.safe_load(open(p)); print(f"{c:24s} {y['experiment']:16s} {y['model']:12s} imgsz={y['train']['imgsz']} batch={y['train']['batch']} sha256={sha(p)[:12]}")
    assert y["dataset_version"] in (DATASET, "e1-yolo-v1"), f"{c}: dataset_version {y['dataset_version']} != {DATASET}"

In [ ]:
import subprocess, time
done = {}
for c, p in CFG.items():
    base = yaml.safe_load(open(p))["experiment"]
    for seed in SEEDS:
        exp = f"{base}_s{seed}"; t0 = time.time()
        r1 = subprocess.run([sys.executable, TRAIN, "--data-dir", DATA_DIR, "--config", p, "--out", "/kaggle/working",
                             "--commit", COMMIT, "--out-name", f"train_{exp}", "--seed", str(seed)])
        ev = f"/kaggle/working/train_{exp}"
        r2 = subprocess.run([sys.executable, ANALYZE, "--data-dir", DATA_DIR, "--manifest", MANIFEST, "--train-evidence", ev,
                             "--out", "/kaggle/working", "--commit", COMMIT, "--out-name", f"analysis_{exp}"])
        done[exp] = {"seed": seed, "config": c, "train_rc": r1.returncode, "analysis_rc": r2.returncode, "minutes": round((time.time() - t0) / 60, 1)}
        print(exp, done[exp])
json.dump({"dataset": DATASET, "fingerprint": fp, "configs": CONFIGS, "seeds": SEEDS, "runs": done}, open("/kaggle/working/w5_seed_runs.json", "w"), indent=2)

## Tự kiểm tra (phải in `PASS`)

In [ ]:
import pandas as pd
rows, ok = [], True
for exp in done:
    st = json.load(open(f"/kaggle/working/train_{exp}/stage_status.json"))
    sa = json.load(open(f"/kaggle/working/analysis_{exp}/analysis_summary.json"))
    m = json.load(open(glob.glob(f"/kaggle/working/train_{exp}/*_val_metrics.json")[0]))["overall"]
    op = sa["stages"].get("operating_points", {})
    good = all(v["status"] == "OK" for v in st.values()) and all(v == "OK" for v in sa["status"].values())
    ok &= good
    rows.append({"model": exp, "ok": good, "mAP50": m["map50"], "mAP50_95": m["map50_95"], "conf_op": op.get("conf_selected_on_val"),
                 "Drone_F1": op.get("Drone_F1"), "ms_img": m["speed_ms_per_image"].get("inference"), "min": done[exp]["minutes"]})
print(pd.DataFrame(rows).to_string(index=False))
with zipfile.ZipFile("/kaggle/working/w5_seed_evidence.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/w5_seed_runs.json", "w5_seed_runs.json")
    for exp in done:
        for d in (f"train_{exp}", f"analysis_{exp}"):
            for p in glob.glob(f"/kaggle/working/{d}/**/*", recursive=True):
                if os.path.isfile(p):
                    z.write(p, os.path.relpath(p, "/kaggle/working"))
print("PASS" if ok else "FAIL - gui w5_seed_runs.json + stage_status.json cho Claude")
p = "/kaggle/working/w5_seed_evidence.zip"; print(os.path.getsize(p) // 1024, "KB  sha256", sha(p))